# 08 - Train Online with GRPO (fresh episode groups)

Each group starts a new episode. All branches copy the same initial environment state
and seed, with only the tokenizer prompt and initial observation as context. Each branch
builds its own history and finishes the entire episode. GRPO compares these full episode
rewards, normalizes them within the group, and applies the clipped policy loss without a
critic. A new group has a new seed and no history from earlier episodes.


In [ ]:
import copy
import json
from functools import partial

import gymnasium as gym
import torch
from mouse_gym import EnvConfig, RepeatTaskTracker, make_env
from procedural_frozenlake import ensure_registered
from repeat_task_gym import RepeatTaskEnv
from torch import nn

from mouse_core import AdamW
from mouse_core.data import (
    Datastore,
    EnvironmentProtocol,
    EpochDataLoader,
    EventLog,
    Tokenizer,
    compose,
    environment_episode_ranges,
)
from mouse_core.models import (
    MLP,
    Backbone,
    DecodeCache,
    Embedding,
    Model,
    MultiEmbedding,
    action_readouts,
    prepare_decode,
    push_model_to_hub,
)
from mouse_core.objectives import (
    GrpoObjective,
    group_relative_advantages,
    sample_discrete_action,
)

# Repositories
# Hugging Face model repo for push_model_to_hub
MODEL_ID = "mouse-example-model-grpo"

# Environment
# number of discrete actions predicted by the policy head
MAX_ACTIONS = 4
# max steps per episode
MAX_STEPS_PER_EPISODE = 24
# inner episodes per outer task
REPEAT_COUNT = 20

# Model
# HF checkpoint for Tokenizer, Embedding, and Backbone
PRETRAINED = "Qwen/Qwen3-0.6B"
HEAD_LAYERS = 1
# final prediction linear's initialization
HEAD_SCALE = 1.0

# Collection and rollout
NUM_ENVS = 2
SEED_OFFSET = 2000000
# G completions per fork
BRANCHES_PER_ENV = 4

# Training
# packed tokens; whole episodes are added until the next one would not fit
TOKEN_BUDGET = 16384
PASSES_PER_COLLECTION = 1
# outer rollout/train cycles
NUM_CYCLES = 2

# Optimizer learning rates
# token embeddings and numeric Fourier features
LR_EMBEDDING = 1e-05
# backbone
LR_BACKBONE = 1e-05
# prediction heads and their input RMSNorms
LR_HEADS = 1e-05

# Runtime
ensure_registered()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Payload names are selected explicitly; the underscore is only a convention.
environment_episode_ranges = partial(
    environment_episode_ranges,
    protocol=EnvironmentProtocol(
        episode_start="_episode_start",
        terminated="_terminated",
        truncated="_truncated",
    ),
)

## Build Environment

Each episode group gets a fresh environment. Branch copies preserve its seed, random
state, and initial observation.


In [ ]:
def make_episode_env(*, seed: int):
    def env_fn():
        lake = gym.make(
            "Procedural-FrozenLake-v1",
            width=8,
            height=8,
            max_episode_steps=MAX_STEPS_PER_EPISODE,
            map_seed=seed,
            slippery_success_rate=1.0,
            permute_obs=True,
            permute_actions=True,
        )
        env = RepeatTaskEnv(
            lake,
            max_task_episodes=REPEAT_COUNT,
            episode_seed_mode="per_task",
            episode_reset_options={"regenerate_map": False},
        )
        return env

    env = make_env(
        EnvConfig(
            name=f"proc_frozenlake_grpo_{seed}",
            seed=seed,
            seed_mode="stream",
            env_fn=env_fn,
            reset_options={"regenerate_map": True},
        )
    )
    return env

## Data Pipeline

Start each group with one reset row at episode step zero. Copy that environment and row
for every branch, then run each branch until the episode ends. Only policy sampling
differs between branches. Both collection and training use each branch's complete
prefix, so behavior probabilities describe the same conditioned policy. Compute
advantages after every branch finishes.


In [ ]:
def episode_separator(ctx):
    ended = ctx.get("inner_terminated", False) or ctx.get("inner_truncated", False)
    return "\n\n" if ended else ""


# Online decode / on-policy buffer: tokenizer
episode_prompt = (
    "Your job is to predict the future sum of rewards in FrozenLake. "
    "Navigate a grid; reach the goal for reward; a hole ends the "
    "episode with none. Each episode lasts at most "
    f"{MAX_STEPS_PER_EPISODE} steps. You have {REPEAT_COUNT} "
    "episodes to solve the task. The grid is permuted, so squares are "
    "not in order; action ids may be remapped, then rendered as a, b, c, d.\n"
    "Strategy: explore; keep a mental map of what has and has not been "
    "explored; avoid holes you have already fallen in; once you have a "
    "path to the goal, repeat it.\n"
    "At each action-request newline, predict the value of all actions. Format:\n"
    "action letter,observation[,r=reward][,terminated][,truncated].\n"
)

tokenizer = Tokenizer(
    tokenizers={"text": PRETRAINED},
    step_field="step_index",
    input_fields=[
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "coordinator",
            "input_field": "_episode_start",
            "format": episode_prompt,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": "episode {field+1}\n",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "agent",
            "input_field": "action",
            "lookup": ["a", "b", "c", "d"],
            "format": "{field}",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "observation",
            "format": ",{field}",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "reward",
            "format": ",r={field:g}",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_terminated",
            "format": ",terminated",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_truncated",
            "format": ",truncated",
        },
        {
            "type": "token",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": "\n",
            "head_output": True,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "format": episode_separator,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
            "missing_value": None,
        },
        {
            "input_field": "_old_log_prob",
            "output_field": "old_log_prob",
            "missing_value": 0.0,
        },
        {
            "input_field": "advantage",
            "missing_value": 0.0,
        },
    ],
)


transform = tokenizer.encode


@torch.no_grad()
def act_once(*, context: list[dict]):
    """Sample one action from context; return (action, old_log_prob)."""
    if not context:
        raise ValueError("act_once requires a non-empty context.")
    # Replay trains complete branches, so behavior must use that same prefix.
    pending = [context]
    inputs, inputs_objective_data = tokenizer.pack_rows(rows=pending)
    out = encoder(inputs, use_cache=False)
    logits = action_readouts(policy_head(out["z"]), inputs)
    actions, log_probs = sample_discrete_action(logits=logits, num_actions=MAX_ACTIONS)
    action = actions[0].cpu().numpy()
    log_probability = float(log_probs[0].item())
    return (action, log_probability)


def rollout_branch(
    *, env, log: EventLog, tracker: RepeatTaskTracker
) -> tuple[list[dict], float]:
    """Complete one branch, recording actions before their environment outputs."""
    context = log.events
    modules.eval()
    while not tracker.ended:
        event_start = len(log)
        episode_start = tracker.output is None or tracker.ended
        episode_index = tracker.next_episode_index
        if episode_start:
            log.event(
                provider="coordinator",
                episode_index=episode_index,
                step_index=(0 if episode_start else tracker.step_index + 1),
                payload={
                    "_episode_start": True,
                },
            )
        if tracker.action_requested:
            action, log_prob = act_once(context=context)
            log.event(
                provider="agent",
                episode_index=episode_index,
                step_index=tracker.step_index + 1,
                payload={
                    "action": action,
                    "_old_log_prob": log_prob,
                },
            )
        else:
            action = None
        output = env.step(action)
        tracker.update(output=output)

        log.event(
            provider="environment",
            episode_index=tracker.episode_index,
            step_index=tracker.step_index,
            payload={
                "observation": tracker.observation,
                **(
                    {"repeat_episode_index": tracker.repeat_episode_index}
                    if tracker.repeat_episode_start
                    else {}
                ),
                **({"inner_terminated": True} if tracker.inner_terminated else {}),
                **({"inner_truncated": True} if tracker.inner_truncated else {}),
                **(
                    {"_q_star": tracker.info["q_star"]}
                    if tracker.action_requested and "q_star" in tracker.info
                    else {}
                ),
                **({"_terminated": True} if tracker.terminated else {}),
                **({"_truncated": True} if tracker.truncated else {}),
                **({"reward": tracker.reward} if tracker.reward != 0 else {}),
                **({"action_requested": True} if tracker.action_requested else {}),
            },
        )
        context.extend(log[event_start:])
    return (context, tracker.tasks[-1].reward)


def fork_group(
    *, env, log: EventLog, tracker: RepeatTaskTracker, branches_per_env: int
) -> tuple[list[list[dict]], list[float]]:
    """Fork the reset environment and its event log into stochastic completions."""
    if len(log) != 2 or tracker.step_index != 0:
        raise ValueError(
            "GRPO groups must start at episode step zero without prior history"
        )
    branches: list[list[dict]] = []
    returns: list[float] = []
    for _ in range(branches_per_env):
        env_g = copy.deepcopy(env)
        log_g = copy.deepcopy(log)
        tracker_g = copy.deepcopy(tracker)
        try:
            rows, ret = rollout_branch(env=env_g, log=log_g, tracker=tracker_g)
            branches.append(rows)
            returns.append(ret)
        finally:
            env_g.close()
    return (branches, returns)


def stamp_advantages_with_prefix_len(
    *,
    branches: list[list[dict]],
    returns: list[float],
    prefix_len: int,
) -> list[list[dict]]:
    adv = group_relative_advantages(rewards=torch.tensor(returns, dtype=torch.float32))
    out: list[list[dict]] = []
    for g, rows in enumerate(branches):
        a = float(adv[g].item())
        new_rows = []
        for t, row in enumerate(rows):
            r = dict(row)
            r["advantage"] = 0.0 if row["step_index"] < prefix_len else a
            r.setdefault("_old_log_prob", 0.0)
            new_rows.append(r)
        out.append(new_rows)
    return out

## Build Encoder and Heads

Call the encoder once, then apply the head(s) to its encodings. Cached inference uses
the same encoder and returns cache and readout-mask metadata. A `ModuleDict` called
`modules` holds references to the online components for checkpointing and mode changes;
it is not called for predictions.


In [ ]:
class Encoder(Model):
    def __init__(
        self, *, pretrained, tokenizer_sources, dtype
    ):
        super().__init__()
        self.embedding = MultiEmbedding(
            embeddings={
                name: Embedding(pretrained=source)
                for name, source in tokenizer_sources.items()
            }
        )
        self.backbone = Backbone(
            pretrained=pretrained,
            dtype=dtype,
        )

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_token_mask"].nonzero(as_tuple=True)[0]
        embed = self.embedding(
            token_ids,
            numeric_values=numeric_values,
            numeric_mask=numeric_mask,
            numeric_field_ids=tensors["numeric_field_ids"],
            numeric_field_names=tokens["numeric_field_names"],
            tokenizer_ids=tensors["tokenizer_ids"],
            tokenizer_names=tokens["tokenizer_names"],
        )
        if use_cache:
            embed, lengths, head_token_ids, _ = prepare_decode(
                embeds=embed,
                tokens=tokens,
                sequence_ids=sequence_ids,
                head_token_mask=tensors["head_token_mask"],
            )
            if cache is None:
                cache = DecodeCache(
                    sessions=(
                        self.backbone.decode_session(
                            batch_size=tokens["num_sequences"]
                        ),
                    )
                )
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape,
                features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        return {
            "z": h,
            "cache": cache,
        }


class PolicyHead(nn.Module):
    def __init__(self, *, hidden_dim, num_actions, num_layers, scale):
        super().__init__()
        self.mlp = MLP(
            in_features=hidden_dim,
            out_features=num_actions,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            scale=scale,
            propagate_gradient=1.0,
        )

    def forward(self, z):
        return self.mlp(z)


encoder = (
    Encoder(
        tokenizer_sources=tokenizer.tokenizer_sources,
        pretrained=PRETRAINED,
        dtype=torch.float32,
    )
    .train()
    .to(device)
)
policy_head = (
    PolicyHead(
        hidden_dim=encoder.backbone.hidden_dim,
        num_actions=MAX_ACTIONS,
        num_layers=HEAD_LAYERS,
        scale=HEAD_SCALE,
    )
    .train()
    .to(device)
)

# A checkpoint container; forward calls use the encoder and heads separately.
modules = nn.ModuleDict({"encoder": encoder, "policy_head": policy_head})

print(encoder)
print(policy_head)

## Collect Fresh Episode Groups

Every collection cycle creates new episodes. No environment or trajectory history
carries over from the previous cycle.


In [ ]:
stores: list[Datastore] = []
branch_advantages = []
next_episode_index = 0

In [ ]:
def run_rollout(*, num_envs: int) -> tuple[int, int]:
    """Collect complete independent branches from fresh episode starts."""
    global stores, next_episode_index, branch_advantages
    modules.eval()
    stores = []
    branch_advantages = []
    n_branches = 0
    for _ in range(num_envs):
        seed_index = next_episode_index
        env = make_episode_env(seed=SEED_OFFSET + seed_index)
        try:
            log = EventLog()
            tracker = RepeatTaskTracker()
            event_start = len(log)
            episode_start = tracker.output is None or tracker.ended
            episode_index = tracker.next_episode_index
            if episode_start:
                log.event(
                    provider="coordinator",
                    episode_index=episode_index,
                    step_index=(0 if episode_start else tracker.step_index + 1),
                    payload={
                        "_episode_start": True,
                    },
                )
            output = env.step(None)
            tracker.update(output=output)

            log.event(
                provider="environment",
                episode_index=tracker.episode_index,
                step_index=tracker.step_index,
                payload={
                    "observation": tracker.observation,
                    **(
                        {"repeat_episode_index": tracker.repeat_episode_index}
                        if tracker.repeat_episode_start
                        else {}
                    ),
                    **({"inner_terminated": True} if tracker.inner_terminated else {}),
                    **({"inner_truncated": True} if tracker.inner_truncated else {}),
                    **(
                        {"_q_star": tracker.info["q_star"]}
                        if tracker.action_requested and "q_star" in tracker.info
                        else {}
                    ),
                    **({"_terminated": True} if tracker.terminated else {}),
                    **({"_truncated": True} if tracker.truncated else {}),
                    **({"reward": tracker.reward} if tracker.reward != 0 else {}),
                    **({"action_requested": True} if tracker.action_requested else {}),
                },
            )
            branches, returns = fork_group(
                env=env, log=log, tracker=tracker, branches_per_env=BRANCHES_PER_ENV
            )
            stamped = stamp_advantages_with_prefix_len(
                branches=branches, returns=returns, prefix_len=1
            )
            for sequence in stamped:
                branch_advantages.append(
                    torch.tensor(
                        [
                            event["advantage"]
                            for event in sequence
                            if event["provider"] == "environment"
                        ]
                    )
                )
                store = Datastore(name=f"grpo_branch_{n_branches}")
                for event in sequence:
                    store.append(
                        data={
                            key: value
                            for key, value in event.items()
                            if key not in ("advantage", "_old_log_prob")
                        }
                    )
                stores.append(store)
                n_branches += 1
            next_episode_index += 1
        finally:
            env.close()
    return (num_envs, n_branches)

## Train

Each cycle makes `PASSES_PER_COLLECTION` shuffled passes through all completed branches,
without replacement. Complete branches are packed within `TOKEN_BUDGET`; the final
partial batch is retained. Advantages and collection probabilities stay fixed during
these passes. There is no critic or delayed model.

The loader returns `inputs` (tokens, embedding features, sequence IDs, and
`head_token_mask`) and `objective_data` (training labels and step mappings).
`objective_data["head_step_ids"][p]` tells the loss which step owns prediction row `p`,
including steps with zero or multiple predictions. Pass this existing mapping to the
objective; no separate prediction-count column is needed. An absent action is
represented by `NaN` in the packed labels.


In [ ]:
objective = GrpoObjective(
    clip_ratio_min=0.8,
    clip_ratio_max=1.2,
    ent_coef=0.01,
    num_actions=MAX_ACTIONS,
)
optimizer = AdamW(
    params=[
        *encoder.parameter_groups(
            lr={"embedding": LR_EMBEDDING, "backbone": LR_BACKBONE}
        ),
        {"params": policy_head.parameters(), "lr": LR_HEADS},
    ],
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)


def run_train(
    *, epochs: int
) -> tuple[torch.Tensor, dict[str, float | torch.Tensor], int]:
    """Visit every completed branch once per epoch."""
    if not stores or sum((len(s) for s in stores)) == 0:
        raise ValueError("Collect complete branches before training")
    modules.train()
    loader = EpochDataLoader(
        stores=stores,
        token_budget=TOKEN_BUDGET,
        epochs=epochs,
        sample_ranges=environment_episode_ranges,
        num_workers=0,
        transform=tokenizer.encode,
    )
    loader.set_sample_objective_fields("advantage", branch_advantages)
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    updates = len(loader)
    try:
        for _ in range(updates):
            inputs, objective_data = loader.next_batch()
            inputs = inputs.to(device)
            objective_data = objective_data.to(device)
            group_id = objective_data["step_sequence_ids"]
            predictions = policy_head(encoder(inputs)["z"])
            losses, metrics = objective(
                objective_data=objective_data.to(device),
                group_id=group_id.to(device),
                head_step_ids=objective_data["head_step_ids"],
                predictions=predictions,
            )
            loss = torch.stack(tuple(losses.values())).sum()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    finally:
        loader.close()
    assert loss is not None, "Training requires at least one optimizer update"
    return (loss, metrics, updates)

## Run

Each cycle collects fresh episode groups, trains on their completed branches, then
discards that batch of trajectories.


In [ ]:
grad_steps = 0
for cycle in range(NUM_CYCLES):
    n_groups, n_branches = run_rollout(num_envs=NUM_ENVS)
    print(f"cycle={cycle} rollout  groups={n_groups}  branches={n_branches}")
    if n_branches == 0:
        continue
    loss, metrics, updates = run_train(epochs=PASSES_PER_COLLECTION)
    grad_steps += updates
    print(
        f"cycle={cycle} train    loss={loss.item():.4f}  policy={metrics['policy_loss']:.4f}  grad_step={grad_steps}",
    )
    stores.clear()
    if device.type == "cuda":
        torch.cuda.empty_cache()

print(f"GRPO demo finished ({grad_steps} optimizer steps, {NUM_CYCLES} cycles).")

## Design Notes

| Setting         | Behavior                                                 |
| --------------- | -------------------------------------------------------- |
| Episode groups  | Fresh environment seed for each group                    |
| Branch start    | Identical copied environment at episode step zero        |
| Initial context | Tokenizer prompt and initial observation only            |
| Branch history  | Accumulates independently within the current episode     |
| Score           | Full episode reward after every branch finishes          |
| Advantage       | Group-normalized episode reward on each branch's actions |
| Loss            | Clipped GRPO policy loss plus entropy; no critic         |

## Save To The Hub

The checkpoint contains the encoder and separate head(s) in `modules`. The model weights
and tokenizer packing specification share `MODEL_ID`. Reconstruct the same component
structure before loading. Checkpoints from the old architecture with a projection are
not compatible with this example.


In [ ]:
modules.eval().to("cpu")
model_url = push_model_to_hub(
    model=modules,
    tokenizer=tokenizer,
    repo_id=MODEL_ID,
    private=False,
    clear=True,
)
print(f"Pushed encoder, heads, and tokenizer to {model_url}")